In [1]:
import functools

import jax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu
import jax.numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P

import numpy as np

from sfp.utils import benchmark, numerics, profile, upload_to_gcs

# jax.config.update('jax_num_cpu_devices', 4)

/home/reed/sfp/.venv/lib/python3.13/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
# m, k, n = 2048, 2048, 1024
m, k, n = 16384, 16384, 8192

k1, k2 = jax.random.split(jax.random.key(0), 2)
inputs = jax.random.normal(k1, (m, k), dtype=jnp.bfloat16)
weights = jax.random.normal(k2, (k, n), dtype=jnp.bfloat16)

In [3]:
num_devices = jax.device_count()
mesh = jax.make_mesh((2, 2), ("x", "y"))
inp_sharding = NamedSharding(mesh, P('x', 'y'))
w_sharding = NamedSharding(mesh, P('x', None))

inputs = jax.device_put(inputs, inp_sharding)
weights = jax.device_put(weights, w_sharding)

/tmp/ipykernel_13227/1985631310.py:2: DeprecationWarning: The default axis_types will change in JAX v0.9.0 to jax.sharding.AxisType.Explicit. To maintain the old behavior, pass `axis_types=(jax.sharding.AxisType.Auto,) * len(axis_names)`. To opt-into the new behavior, pass `axis_types=(jax.sharding.AxisType.Explicit,) * len(axis_names)
  mesh = jax.make_mesh((2, 2), ("x", "y"))


inputs are size 2048, 2048 -> bf16:: 2 bytes * 2048 * 2048 = ~8MB
weights are size 2048, 1024 -> bf16:: 2 bytes * 2048 * 1024 = ~4MB

inputs sharded along x and y -> $Inp[I_{X}, J_{Y}]$

weights sharded along x -> $W[J_{X}, K]$

Each device has N elements per array:
  - inputs
    - (2048 / 2) * (2048 / 2) * 2bytes
    - ~2MB
  - weights
    - (2048 / 2) * 1024 * 2bytes
    - ~2MB

The contracting dimension is sharded in both inputs and weights, along different axes.
Need to handle that with collectives; AG/AR

In [4]:
mesh.devices

array([[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
        TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0)],
       [TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0),
        TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0)]],
      dtype=object)

In [5]:
mesh.device_ids

array([[0, 1],
       [3, 2]])

In [6]:
for i in inputs.addressable_shards:
    dev = i.device
    idx = i.index
    print(f"Device:{dev}\nID:{idx}")

Device:TPU_0(process=0,(0,0,0,0))
ID:(slice(0, 8192, None), slice(0, 8192, None))
Device:TPU_1(process=0,(1,0,0,0))
ID:(slice(0, 8192, None), slice(8192, 16384, None))
Device:TPU_3(process=0,(1,1,0,0))
ID:(slice(8192, 16384, None), slice(0, 8192, None))
Device:TPU_2(process=0,(0,1,0,0))
ID:(slice(8192, 16384, None), slice(8192, 16384, None))


In [7]:
for i in weights.addressable_shards:
    dev = i.device
    idx = i.index
    print(f"Device:{dev}\nID:{idx}")

Device:TPU_0(process=0,(0,0,0,0))
ID:(slice(0, 8192, None), slice(None, None, None))
Device:TPU_1(process=0,(1,0,0,0))
ID:(slice(0, 8192, None), slice(None, None, None))
Device:TPU_3(process=0,(1,1,0,0))
ID:(slice(8192, 16384, None), slice(None, None, None))
Device:TPU_2(process=0,(0,1,0,0))
ID:(slice(8192, 16384, None), slice(None, None, None))


In [8]:
jax.debug.visualize_array_sharding(inputs)

                        
                        
   TPU 0       TPU 1    
                        
                        
                        
                        
                        
   TPU 3       TPU 2    
                        
                        
                        

In [9]:
jax.debug.visualize_array_sharding(weights)

            
            
  TPU 0,1   
            
            
            
            
            
  TPU 2,3   
            
            
            

In [10]:
def jax_matmul(x: jax.Array, y: jax.Array) -> jax.Array:
    return jnp.matmul(x, y)

@functools.partial(
    jax.shard_map,
    mesh=mesh,
    in_specs=(P('x', 'y'), P('x', None)),
    out_specs=P('x', None),
    check_vma=False
)
def xla_matmul_1(input_shard: jax.Array, w_shard: jax.Array) -> jax.Array:
    # First we want to all_gather the data
    with jax.named_scope('all_gather(s)'):
        input_full = jax.lax.all_gather(input_shard, 'y', axis=1, tiled=True)
        w_full = jax.lax.all_gather(w_shard, 'x', axis=0, tiled=True) # gather w along x
    # Then we want to compute on the data
    # Since we did two full all gathers to start, regular GEMMs
    with jax.named_scope('dot'):
        local_out = input_full @ w_full
    return local_out

In [11]:
ref = jax_matmul(inputs, weights)

In [12]:
ref

Array([[92, -233, -7, ..., -17.75, 37, -173],
       [-75, 59, 85.5, ..., -80.5, 73, -66.5],
       [53, -19.25, 24.75, ..., -58.25, -175, 206],
       ...,
       [44.75, -113, 52, ..., -116.5, 150, 8.875],
       [-73, 284, -134, ..., -8, -376, 124.5],
       [-396, -89.5, -3.21875, ..., 92.5, -27.75, 11.25]], dtype=bfloat16)

In [13]:
benchmark(jax_matmul, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:       12.221 ms
  median:     12.218 ms
  stdev:       0.015 ms
  min:        12.201 ms
  max:        12.281 ms
  p95:        12.245 ms
  p99:        12.265 ms

In [14]:
# numerics.compare(jax_matmul(inputs, weights), xla_matmul_1(inputs, weights), rtol=1e-2, atol=1e-2, region_grid=(2,2))
jitted = jax.jit(xla_matmul_1)
test = xla_matmul_1(inputs, weights)
benchmark(jitted, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:       17.617 ms
  median:     17.616 ms
  stdev:       0.007 ms
  min:        17.602 ms
  max:        17.633 ms
  p95:        17.629 ms
  p99:        17.632 ms

In [15]:
numerics.compare(ref, test)

NumericsResult(FAIL)
  shape:     (16384, 8192)
  max_diff:  4.000000
  mean_diff: 0.167969
  median:    0.000000
  % > 0.1: 33.09%
  worst at (5, 7889): ref=-592.0000, test=-588.0000

In [16]:
out = jax_matmul(inputs, weights)
jax.debug.visualize_array_sharding(out)

            
            
  TPU 0,1   
            
            
            
            
            
  TPU 2,3   
            
            
            

In [91]:
jax_matmul_compiled = jax.jit(jax_matmul)
jmc = jax_matmul_compiled(inputs, weights)
jmc.block_until_ready()

with jax.profiler.trace('./traces/naive_matmul'):
    result = jax_matmul_compiled(inputs, weights)
    result.block_until_ready()

"""
    %all-reduce = bf16[1024,1024]{1,0:T(8,128)(2,1)} all-reduce(bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %fusion), channel_id=2,
    replica_groups=[2,2]<=[4], use_global_device_ids=true, to_apply=%add.clone

    ^^ {1,0:T(8,128)(2,1)} ; row major** -- read these as most minor to most major (or right to left)

    %fusion = bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} fusion(bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %copy-done,
    bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %collective-permute-done), kind=kOutput, calls=%fused_computation

    %all-reduce = bf16[1024,1024]{1,0:T(8,128)(2,1)} all-reduce(bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %fusion), channel_id=2,
    replica_groups=[2,2]<=[4], use_global_device_ids=true, to_apply=%add.clone
"""


# xm1 = jax.jit(xla_matmul_1)
# result = xm1(inputs, weights)
# result.block_until_ready()

# with jax.profiler.trace(TRACES_DIR):
#     result = xm1(inputs, weights)
#     result.block_until_ready()

'\n    %all-reduce = bf16[1024,1024]{1,0:T(8,128)(2,1)} all-reduce(bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %fusion), channel_id=2,\n    replica_groups=[2,2]<=[4], use_global_device_ids=true, to_apply=%add.clone\n\n    ^^ {1,0:T(8,128)(2,1)} ; row major** -- read these as most minor to most major (or right to left)\n\n    %fusion = bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} fusion(bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %copy-done,\n    bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %collective-permute-done), kind=kOutput, calls=%fused_computation\n\n    %all-reduce = bf16[1024,1024]{1,0:T(8,128)(2,1)} all-reduce(bf16[1024,1024]{1,0:T(8,128)(2,1)S(1)} %fusion), channel_id=2,\n    replica_groups=[2,2]<=[4], use_global_device_ids=true, to_apply=%add.clone\n'

In [15]:
@functools.partial(
    jax.shard_map,
    mesh=mesh,
    in_specs=(P('x', 'y'), P('x', None)),
    out_specs=P('x', None),
    check_vma=False
)
def xla_matmul_2(input_shard: jax.Array, weight_shard: jax.Array) -> jax.Array:
    """
    This time, we want to make the computation a little more efficient than
    stacking the two all gathers at the beginning of the kernel

    All Reduce at the end over partial sums
    """

    # All gather the weights over x so that each device contains full copy
    w_full = jax.lax.all_gather(weight_shard, 'x', axis=0, tiled=True)

    """
    Once we have the full weight matrix on each device, we know that our y_idx
    will let us pluck out columns of inputs, but we need to make sure
    that we're plucking out the appropriate rows of the weights
    """
    y_idx = jax.lax.axis_index('y')
    y_size = jax.lax.axis_size('y')
    k, n = w_full.shape
    k_block = k // y_size


    # Using the y-ring axis to determined which col stripe of weights to compute locally
    w_slice = jax.lax.dynamic_slice(w_full, (y_idx * k_block, 0), (k_block, n))
    local_out = input_shard @ w_slice
    # All Reduce over the y-ring to accumulate partial results
    out = jax.lax.psum(local_out, 'y')
    return out

In [22]:
jax_matmul_compiled_2 = jax.jit(xla_matmul_2)
jmc_2 = jax_matmul_compiled_2(inputs, weights)
jmc_2.block_until_ready()

with jax.profiler.trace('./traces/matmul2'):
    result = jax_matmul_compiled_2(inputs, weights)
    result.block_until_ready()

In [16]:
@functools.partial(
    jax.shard_map,
    mesh=mesh,
    in_specs=(P('x', 'y'), P('x', None)),
    out_specs=P('x', None)
)
def xla_matmul_3(input_shard: jax.Array, weight_shard: jax.Array) -> jax.Array:
    """
    Use some higher precision numerics to demonstrate accumulation order (fp32)
    """
    w_full = jax.lax.all_gather(weight_shard, 'x', axis=0, tiled=True)
    y_idx = jax.lax.axis_index('y')
    y_size = jax.lax.axis_size('y')
    k, n = w_full.shape
    k_block = k // y_size
    w_slice = jax.lax.dynamic_slice(w_full, (y_idx * k_block, 0), (k_block, n))

    local_out = jax.lax.dot_general(
        input_shard, w_slice,
        dimension_numbers=(((1,), (0,)), ((), ())),
        precision=jax.lax.Precision.HIGHEST,
        preferred_element_type=jnp.float32,
    )
    out = jax.lax.psum(local_out, 'y')
    return out

In [17]:
jnp.allclose(xla_matmul_3(inputs, weights), jax_matmul(inputs, weights), rtol=1e-2, atol=1e-2)

Array(False, dtype=bool)

In [18]:
xla_matmul_3(inputs, weights)

Array([[  91.71906  , -233.11984  ,   -7.083885 , ...,  -17.664907 ,
          36.83552  , -172.64694  ],
       [ -75.25015  ,   58.6091   ,   85.601135 , ...,  -80.40517  ,
          73.22124  ,  -66.55117  ],
       [  52.96561  ,  -19.443443 ,   24.725124 , ...,  -58.151703 ,
        -174.73862  ,  205.62843  ],
       ...,
       [  44.641594 , -112.32719  ,   52.143745 , ..., -116.69063  ,
         150.0973   ,    8.866718 ],
       [ -72.57734  ,  283.12402  , -133.44708  , ...,   -7.9149017,
        -375.95526  ,  124.36682  ],
       [-395.974    ,  -89.63108  ,   -3.249035 , ...,   92.28265  ,
         -27.732243 ,   11.188175 ]], dtype=float32)

In [19]:
@functools.partial(
  jax.shard_map,
  mesh=mesh,
  in_specs=(P('x', 'y'), P('x', None)),
  out_specs=P('x', None)
)
def pperm_xla_matmul(inp, weight):
  """
  In this scenario, we realize that we actaully don't need to send the full
  data both ways. That is, in our simple square, we recognize that only the
  off-diagonal devices _need_ the data
  """
  x_idx = jax.lax.axis_index('x')
  # Swap the shards along the x mesh axis
  weight_other = jax.lax.ppermute(weight, 'x', perm=[(0, 1), (1, 0)])

  # Build the shards appropriately
  w_full = jax.lax.cond(
    x_idx == 0,
    lambda _: jax.lax.concatenate([weight, weight_other], dimension=0),
    lambda _: jax.lax.concatenate([weight_other, weight], dimension=0),
    operand=None
  ) # (K, N)

  y_idx  = jax.lax.axis_index('y')
  y_size = jax.lax.axis_size('y')

  k, n = w_full.shape
  k_block = k // y_size

  w_slice = jax.lax.dynamic_slice(w_full, (y_idx * k_block, 0), (k_block, n))
  local_out = inp @ w_slice
  out = jax.lax.psum(local_out, 'y')
  return out

In [26]:
jax_matmul_compiled_4 = jax.jit(pperm_xla_matmul)
jmc_4 = jax_matmul_compiled_4(inputs, weights)
jmc_4.block_until_ready()

with jax.profiler.trace('./traces/pperm_matmul'):
    result = jax_matmul_compiled_4(inputs, weights)
    result.block_until_ready()

In [20]:
jnp.allclose(pperm_xla_matmul(inputs, weights), jax_matmul(inputs, weights), rtol=1e-2, atol=1e-2)

Array(True, dtype=bool)

In [ ]:
"""
Let's recall what we've learned so far --

When needed to perform an all gather on the reduction axis of our weights
to remove the sharding over X

Then, we compute local MatMuls (slicing out the appropriate data) between
the shard local inputs and the full weights
- Recall, we have the full weights after AG, so we need to slice out the
  appropriate chunks of W for our computation

These MatMuls are accumulators -> They finally need to be all reduced
over Y. The desired out sharding is ('x', None), so when we do an AR
over the Y axis, we are sharing the partial results
"""

"""
Let's sketch out the algorithm we care about --

We have 2 arrays distributed over 4 devices
  - 1/4 of inputs on each device
  - 1/2 of weights of each device

We want to efficiently compute this distributed matmul over the devices

We know that the contracting dims are sharded differently

So there will need to be some comms to unshard so that we have the
whole array in the right place. HOWEVER, we may also be able to get
away with ppermute to simple pass _results_ after compute is finished

Here are the kernels we may want to try
  - Simple matmul with lax collectives inserted in the right spots
  - MatMul with handrolled collectives (still AG to start, then AR)
  - ppermute
    - Issue async DMA
    - Run local compute; stash in accumulator
    - 
    - How much latency can we hide here?
      - If the DMAs are fast/slow?
      - How do we _reason_ about these tradeoffs

Start with 2x2 case; don't worry too much about abstracting things out
  - Then extend to larger configurations + abstractions
  - How do we think about the work we're doing?
  - Where are the opportunities to show different edge cases?
  - Where do our assumptions break down?
  - emit_pipeline
  - kernel schedule?
  - What happens when we run this on Trillium?
    - What changes?
"""

In [ ]:
"""
  Here's what happens:
  - We have 2 arrays in HBM
  - Need to all_gather weights over x, now each device has fully copy of x
  - Then do local compute
  - All Reduce the local compute to get the correct results
"""

#NOTE: THIS IS THE CASE WHERE WE SIMPLY REPLACE jax.dot/jax.matmul/x @ y in xla_matmul3
def simple_matmul(x_ref, y_ref, o_ref, scratch_ref, *, n_steps):
  # Zero scratch buffer
  with jax.named_scope('Zero Scratch Buffer'):
    @pl.when(pl.program_id(2) == 0)
    def _init_scratch():
      scratch_ref[...] = jnp.zeros_like(scratch_ref)

  # Compute dot
  with jax.named_scope('Compute GEMM'):
    scratch_ref[...] += jnp.dot(
      x_ref[...],
      y_ref[...],
      preferred_element_type=jnp.float32
    )

  # Flush to HBM
  with jax.named_scope('Flush to HBM'):
    @pl.when(pl.program_id(2) == n_steps - 1)
    def _flush_scratch():
      o_ref[...] = scratch_ref[...].astype(o_ref.dtype)


def make_matmul(
  x: jax.Array,
  y: jax.Array,
  *,
  bm: int = 128,
  bk: int = 128,
  bn: int = 128,
):
  m, k = x.shape
  _, n = y.shape

  grid_spec = pltpu.PrefetchScalarGridSpec(
    num_scalar_prefetch=0,
    grid=(m//bm, n//bn, k//bk),
    in_specs=[
      pl.BlockSpec((bm, bk), lambda i,j,k: (i, k)),
      pl.BlockSpec((bk, bn), lambda i,j,k: (k, j))
    ],
    out_specs=pl.BlockSpec((bm, bn), lambda i, j, k: (i, j)),
    scratch_shapes=[pltpu.VMEM((bm, bn), jnp.float32)]
  )

  return pl.pallas_call(
    functools.partial(simple_matmul, n_steps=k//bk),
    grid_spec=grid_spec,
    out_shape=jax.ShapeDtypeStruct((m, n), dtype=jnp.bfloat16)
  )(x, y)

In [22]:
def distributed_gemm_kernel1(inputs, weights):
  w_full = jax.lax.all_gather(weights, 'x', axis=0, tiled=True)
  
  y_idx = jax.lax.axis_index('y')
  y_sz = jax.lax.axis_size('y')
  k, n = w_full.shape
  k_block = k // y_sz
  w_slice = jax.lax.dynamic_slice(w_full, (y_idx * k_block, 0), (k_block, n))

  # Now let's use our matmul kernel in place jax.dot
  local_out = make_matmul(inputs, w_slice, bm=512, bk=1024, bn=1024)
  return jax.lax.psum(local_out, 'y')

In [23]:
dgk1 = jax.jit(
    jax.shard_map(
    distributed_gemm_kernel1,
    mesh=mesh,
    in_specs=(P('x', 'y'), P('x', None)),
    out_specs=P('x', None),
    check_vma=False
))

In [26]:
dgk_compiled = dgk1.lower(inputs, weights).compile({'xla_enable_transpose_trace': True})
dgk = dgk_compiled(inputs, weights)
dgk.block_until_ready()

with jax.profiler.trace('./traces/dgk'):
    result = dgk_compiled(inputs, weights)
    result.block_until_ready()

In [27]:
benchmark(dgk1, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:       12.696 ms
  median:     12.696 ms
  stdev:       0.008 ms
  min:        12.679 ms
  max:        12.716 ms
  p95:        12.713 ms
  p99:        12.715 ms

In [21]:
ref = jax_matmul(inputs, weights)
test = dgk1(inputs, weights)

numerics.compare(ref, test, atol=1e-2, rtol=1e-2, region_grid=(2,2))

NumericsResult(FAIL)
  shape:     (8192, 4096)
  max_diff:  2.000000
  mean_diff: 0.000010
  median:    0.000000
  % > 0.1: 0.00%
  worst at (5573, 2968): ref=-158.0000, test=-156.0000
  regions (2x2):
    [0,0]: mean=0.0000, %bad=0.0%
    [0,1]: mean=0.0000, %bad=0.0%
    [1,0]: mean=0.0000, %bad=0.0%
    [1,1]: mean=0.0000, %bad=0.0%

In [24]:
def all_gather_kernel_1D(
  input_ref, output_ref,
  local_send_sem, send_sem, recv_sem, 
):
  """
  input_ref: shard local data
  output_ref: out shard

  local_send_sem: allocates a semaphore for the local HBM copy
  send_sem: semaphore for the RDMA push
  recv_sem: semaphore for our local data
  """
  # TODO: Barrier

  pid = pl.program_id(0)
  shard_height = input_ref.shape[0]
  shard_width = input_ref.shape[1]

  # Get neighbors
  x_ring = jax.lax.axis_size('x')
  this_device_x = jax.lax.axis_index('x')
  this_device_y = jax.lax.axis_index('y')
  right_device_x = jax.lax.rem(this_device_x + 1, x_ring)

  # Recall: This is the _destination_ copy slot
  """
  Imagine that you're in a length 4 ring at position 0
    - Iter 1: You pass your data (say row 0) to the right
      - You receieve row 1 from your right, and row 3 from your left
    - Iter 2: You have (0, 1, X, 3)
      - If you're sending right; You need to send row 3
      - B/c Device 1 has (0, 1, 2, X)
      - Basically, imagine passing what you just received from your left, to the right
  """
  copy_slot_xright = jax.lax.rem(this_device_x - pid, x_ring)
  copy_slot_xleft = jax.lax.rem(this_device_x + pid, x_ring)

  # We're just copying within our HBM to a bigger HBM memory
  with jax.named_scope("Local HBM Copy"):
    @pl.when(pl.program_id(0) == 0)
    def _copy_local_to_local():
      local_hbm_copy = pltpu.make_async_copy(
        src_ref=input_ref,
        dst_ref=output_ref.at[pl.ds(this_device_x * shard_height, shard_height), :],
        sem=local_send_sem
      )
      with jax.named_scope("Local Copy Start"):
        local_hbm_copy.start()
      with jax.named_scope("Local Copy Wait"):
        local_hbm_copy.wait()

  right_dma = pltpu.make_async_remote_copy(
    src_ref=output_ref.at[pl.ds(copy_slot_xright * shard_height, shard_height), :],
    dst_ref=output_ref.at[pl.ds(copy_slot_xright * shard_height, shard_height), :],
    send_sem=send_sem,
    recv_sem=recv_sem,
    device_id=(right_device_x, this_device_y),
    device_id_type=pltpu.DeviceIdType.MESH,
  )

  with jax.named_scope("Right DMA Start"):
    right_dma.start()
  with jax.named_scope("Right DMA Wait"):
    right_dma.wait()


def make_ag(x, interpret: None | bool = None):
  if not interpret:
    platform = jax.devices()[0].platform
    if platform == 'tpu':
      interpret = False
    else:
      # ignoring gpu for now
      interpret=True

  rows, cols = x.shape
  x_size = jax.lax.axis_size('x')

  grid_spec_ag1d = pltpu.PrefetchScalarGridSpec(
  num_scalar_prefetch=0,
  grid=(1,),
  in_specs=[
    # Our input reference is just our big tensor in HBM
    pl.BlockSpec(memory_space=pl.ANY)
  ],
  # Our output reference will be _another_ big tensor in HBM
  out_specs=pl.BlockSpec(memory_space=pl.ANY),
  scratch_shapes=(
    [pltpu.SemaphoreType.DMA] * 2 # local_copy_op, send_sem
    + [pltpu.SemaphoreType.DMA] * 1 # These are our recv_sems. For 2x2, we only need 1 of them
  )
)

  out_shape_ag1d=jax.ShapeDtypeStruct((rows * x_size, cols), dtype=jnp.bfloat16)
  
  return pl.pallas_call(
    all_gather_kernel_1D,
    grid_spec=grid_spec_ag1d,
    out_shape=out_shape_ag1d,
    interpret=interpret
  )(x)

In [25]:
xla_ag = jax.jit(
    jax.shard_map(
        lambda x: jax.lax.all_gather(x, 'x', tiled=True),
        mesh=mesh, in_specs=P('x', None), out_specs=P('x', None))
)(weights)

ag1d = jax.jit(
  jax.shard_map(
      make_ag,
      mesh=mesh,
      in_specs=P('x', None),
      out_specs=P('x', None),
      check_vma=False
  )
)(weights)

In [26]:
jnp.allclose(xla_ag, ag1d, atol=1e-2, rtol=1e-2)

Array(True, dtype=bool)

In [27]:
"""
NOTE: When I was trying to directly use 
`output_ref[...] = local_hbm_ref[...] + output_ref[...]` it was causing an error.
> ValueError: Loads are only allowed on VMEM and SMEM references. ANY memory space can only be accessed using async_copy.

Reason:
output_ref is marked at pl.ANY. Only VMEM/SMEM are addressable by compute units (ref[...] compiles to loads/stores)
Off-chip memory (HBM; pl.ANY) accessible via DMAs, not loads/stores
"""

# TODO: Make this kernel work for bigger problem sizes
def all_reduce_kernel_1D(
    local_hbm_ref, output_ref,
    send_sem, recv_sem, copy_sem,
    local_scratch, recv_scratch 
):
    """
    Right now, here's what we'll have --
    We have all-gathered the full weight tensor into each device's HBM
    THEN: we will have compute a GEMM over the device-local chunks
    We need to all-reduce over the Y AXIS at the end so that
    all the data is in the right place/on the right device

    The data needs to go from P('x', NONE) -> P('x', None)
    Basically -> We did GEMMs on inputs[0:M/2, 0:N/2] @ weights[0:M/2,N], ...

    We need to sum those row stipes over the y-axis, and everything will
    be good to go

    This will be _SLOW_ for now because of the HBM traffic
    """

    y_ring = jax.lax.axis_size('y')
    this_device_x = jax.lax.axis_index('x')
    this_device_y = jax.lax.axis_index('y')
    right_device_y = jax.lax.rem(this_device_y + 1, y_ring)

    with jax.named_scope("Create Local Copy"):
        local_copy = pltpu.make_async_copy(
        src_ref=local_hbm_ref,
        dst_ref=local_scratch,
        sem=copy_sem
        )

    with jax.named_scope("Local Copy"):
        local_copy.start()
        local_copy.wait()

    # This will copy our HBM tile into either:
    #  - Remote HBM Tile
    #    - Right now our GEMM works on HBM, so this will be easier temorarily
    #  - Remote VMEM tile (Memory pressure)
    send_ref = local_scratch
    for _ in range(y_ring - 1):
        right_dma = pltpu.make_async_remote_copy(
            src_ref=send_ref,
            dst_ref=recv_scratch,
            send_sem=send_sem,
            recv_sem=recv_sem,
            device_id=(this_device_x, right_device_y),
            device_id_type=pltpu.DeviceIdType.MESH
        )

        with jax.named_scope("Right DMA Start"):
            right_dma.start()
        # with jax.named_scope("Local Copy Wait"):
        #     local_copy.wait()
        with jax.named_scope("Right DMA Wait"):
            right_dma.wait()

        # output_ref[...] = local_hbm_ref[...] + output_ref[...]
        # Add in VMEM, write back to HBM
        with jax.named_scope("Add Local Data to Remote Data"):
            local_scratch[...] = local_scratch[...] + recv_scratch[...]

        send_ref = recv_scratch

    out_copy = pltpu.make_async_copy(
          src_ref=local_scratch,
          dst_ref=output_ref,
          sem=copy_sem
      )
    
    with jax.named_scope("Copy Out"):
        out_copy.start()
        out_copy.wait()

"""
Notice how you can also achieve the automatic HBM pipelining with blockspecs
as you might with a GEMM

def all_reduce_kernel_1D(
    local_ref,      # VMEM (auto-copied from HBM by Pallas)
    output_ref,     # VMEM (auto-copied to HBM by Pallas)
    recv_scratch,   # VMEM scratch
    send_sem, recv_sem
):
    y_ring = jax.lax.axis_size('y')
    this_device_x = jax.lax.axis_index('x')
    this_device_y = jax.lax.axis_index('y')
    right_device_y = jax.lax.rem(this_device_y + 1, y_ring)

    right_dma = pltpu.make_async_remote_copy(
        src_ref=local_ref,
        dst_ref=recv_scratch,
        send_sem=send_sem,
        recv_sem=recv_sem,
        device_id=(this_device_x, right_device_y),
        device_id_type=pltpu.DeviceIdType.MESH
    )
    right_dma.start()
    right_dma.wait()

    output_ref[...] = local_ref[...] + recv_scratch[...]
    """


def make_ar(input_array, bm=1024, bn=1024):
    m_local, n_local = input_array.shape

    grid_spec_ar1d=pltpu.PrefetchScalarGridSpec(
    num_scalar_prefetch=0,
    grid=(m_local // bm, n_local // bn),
    in_specs=[
        pl.BlockSpec((bm, bn), lambda i, j: (i, j)),
    ],
    out_specs=pl.BlockSpec((bm, bn), lambda i, j: (i, j)),
    scratch_shapes=(
        [pltpu.SemaphoreType.DMA] * 3 # send_sem, recv_sem, copy_sem
        + [pltpu.VMEM((bm, bn), jnp.bfloat16)] # local_scratch
        + [pltpu.VMEM((bm, bn), jnp.bfloat16)] # recv_scratch
        )
    )

    out_shape = jax.ShapeDtypeStruct(input_array.shape, input_array.dtype)

    return pl.pallas_call(
        all_reduce_kernel_1D,
        grid_spec=grid_spec_ar1d,
        out_shape=out_shape,
    )(input_array)

In [28]:
def slow_ag_gemm_ar_kernel(inputs, weights):
    y_idx = jax.lax.axis_index('y')
    y_size = jax.lax.axis_size('y')
    a = make_ag(weights)

    # We need to remember that we're inside a shard map
    # _WHERE_ we insert these shape operations matters because
    # under shard_map we're looking at device local data
    k_full, n = a.shape
    k_block = k_full // y_size

    a_slice = jax.lax.dynamic_slice(a, (y_idx * k_block, 0), (k_block, n))
    b = make_matmul(inputs, a_slice, bm=512, bk=1024, bn=1024)
    c = make_ar(b)
    return c

In [29]:
"""
ValueError: Cannot signal on a non-()-shaped semaphore: (1024, 1024)
-> Don't forget that kernel arg order _matters_ wrt scratch shapes
"""

sagak = jax.jit(
    jax.shard_map(
        slow_ag_gemm_ar_kernel,
        mesh=mesh,
        in_specs=(P('x', 'y'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False
    )
)

sagak(inputs, weights)

Array([[92, -233, -7, ..., -17.75, 37, -173],
       [-75, 59, 85.5, ..., -80.5, 73, -66.5],
       [53, -19.25, 24.75, ..., -58.25, -175, 206],
       ...,
       [44.75, -113, 52, ..., -116.5, 150, 8.875],
       [-73, 284, -134, ..., -8, -376, 124.5],
       [-396, -89.5, -3.21875, ..., 92.5, -27.75, 11.25]], dtype=bfloat16)

In [30]:
# NOTE: These benchmarks look dominated by overhead
# Need to rely on profile for accurate information
benchmark(sagak, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:       13.371 ms
  median:     13.372 ms
  stdev:       0.008 ms
  min:        13.351 ms
  max:        13.390 ms
  p95:        13.383 ms
  p99:        13.388 ms

In [137]:
a = jax_matmul(inputs, weights)
b = sagak(inputs, weights)

numerics.compare(a, b)

NumericsResult(FAIL)
  shape:     (16384, 8192)
  max_diff:  2.000000
  mean_diff: 0.000021
  median:    0.000000
  % > 0.1: 0.00%
  worst at (83, 2034): ref=-268.0000, test=-266.0000

In [31]:
opts = jax.profiler.ProfileOptions()
opts.advanced_configuration = {
    "tpu_trace_mode": "TRACE_COMPUTE_AND_SYNC",
}

In [36]:
sagak_compiled = sagak.lower(inputs, weights).compile({'xla_enable_transpose_trace': True})
result = sagak_compiled(inputs, weights)
result.block_until_ready()

with jax.profiler.trace('./traces/sagak', profiler_options=opts):
    result = sagak_compiled(inputs, weights)
    result.block_until_ready()

"""
113 us vs 208 us, slow but workable
"""

'\n113 us vs 208 us, slow but workable\n'

In [ ]:
"""
Basically --
The way we organize the grid for communications is a decision
  - Rings, ranges, etc.
  - This trades off bandwidth/latency


# Ring
grid = (ring_size - 1,)

# Recursive doubling
grid = (log2(ring_size),)

# Direct
grid = (1,)  # but more complex RDMA pattern
"""

In [ ]:
"""
Here's something fun --

NotImplementedError: Meshes with more than 1 named
dimension not implemented in dma_start_p

Pallas's remote DMA primitives currently only support 1D meshes

https://github.com/jax-ml/jax/blob/main/jax/_src/pallas/mosaic/primitives.py
"""

In [ ]:
# EXTEND THE KERNEL TO BE MORE GENERAL THAN 2x2 ; SCALE UP TO 4x4 GRID

In [32]:
"""
On each local device, we have 2 arrays sitting there -> inputs + weights
On kernel start:
  - Barrier sync to get everyone on the same stage
    - Might could relax this constraint?
  - Issue remote DMAs along the ... ring to all gather the weights
  - Once received
    - Compute local dots
    - Accumulate
  - Finally
    - All Reduce over the y ring to conform shmap shape


Additional notes:
  RECALL: n_hopes = grid (ring_len) - 1 -> I think this is just using one ICI link though
  What happens at n = 1? 2? 3?
  n=1:
    - You send right, receive left (this kernel is only looking at 1D case rn)
    - At (0,0): You send to (1,0) the chunk starting at 0 * row_height
    - You receive chunks from (1,0) and (3, 0)
    - You have [x, x, O, x] ; need idx 2 * row_height
    - r_neighbor needs -> [x, x, x, O]
  n=2:
    - This would be the epilogue already w/ 2 ICI links**
    - You send r_neighbor+1 chunk ; could probably push this up to initial mappings
"""

def all_gather_kernel_bidi(
  input_ref, output_ref,
  local_send_sem, send_sem_right, send_sem_left,
  recv_sem_right, recv_sem_left
):
  """
  input_ref: shard local data
  output_ref: out shard

  local_send_sem: allocates a semaphore for the local HBM copy
  send_sem: semaphore for the RDMA push
  recv_sem: semaphore for our local data
  """

  pid = pl.program_id(0)
  
  shard_height = input_ref.shape[0]

  this_device_x = jax.lax.axis_index('x')
  this_device_y = jax.lax.axis_index('y')
  x_ring = jax.lax.axis_size('x')
  right_device_x = jax.lax.rem(this_device_x + 1, x_ring)
  # Don't want negatives
  left_device_x = jax.lax.rem(this_device_x - 1 + x_ring, x_ring)

  # This accounts for the offset when data is being sent both ways
  copy_slot_right = jax.lax.rem(this_device_x - pid + x_ring, x_ring)
  copy_slot_left = jax.lax.rem(this_device_x + pid, x_ring)


  # PERFORM INITIAL ASYNC COPY FROM OUR HBM TO OUR HBM
  # We're just copying within our HBM to a bigger HBM memory
  # XLA liveness should handle malloc/free the _INPUT_ tensor once the AG completes
  @pl.when(pid == 0)
  def _prologue_sends():
    local_copy = pltpu.make_async_copy(
    src_ref=input_ref,
    dst_ref=output_ref.at[pl.ds(this_device_x * shard_height, shard_height), :],
    sem=local_send_sem
  )

    local_copy.start()
    local_copy.wait()

  right_dma = pltpu.make_async_remote_copy(
    # Next kernel iter depends on completion of left/right DMAs
    src_ref=output_ref.at[pl.ds(copy_slot_right * shard_height, shard_height), :],
    dst_ref=output_ref.at[pl.ds(copy_slot_right * shard_height, shard_height), :],
    send_sem=send_sem_right,
    recv_sem=recv_sem_right,
    device_id=(right_device_x, this_device_y),
    device_id_type=pltpu.DeviceIdType.MESH,
  )

  left_dma = pltpu.make_async_remote_copy(
    src_ref=output_ref.at[pl.ds(copy_slot_left * shard_height, shard_height), :],
    dst_ref=output_ref.at[pl.ds(copy_slot_left * shard_height, shard_height), :],
    send_sem=send_sem_left,
    recv_sem=recv_sem_left,
    device_id=(left_device_x, this_device_y),
    device_id_type=pltpu.DeviceIdType.MESH
  )

  right_dma.start()
  left_dma.start()
  right_dma.wait()
  left_dma.wait()
  
  @pl.when(pl.program_id(0) == pl.num_programs(0) - 1)
  def _epilogue():
    right_dma = pltpu.make_async_remote_copy(
      # Next kernel iter depends on completion of left/right DMAs
      src_ref=output_ref.at[pl.ds(copy_slot_right * shard_height, shard_height), :],
      dst_ref=output_ref.at[pl.ds(copy_slot_right * shard_height, shard_height), :],
      send_sem=send_sem_right,
      recv_sem=recv_sem_right,
      device_id=(right_device_x, this_device_y),
      device_id_type=pltpu.DeviceIdType.MESH,
    )

    right_dma.start()
    right_dma.wait()

def make_ag(x):
  mesh_len = jax.lax.axis_size('x')
  # Cheating only a little
  m, k = x.shape

  grid_spec = pltpu.PrefetchScalarGridSpec(
  num_scalar_prefetch=0,
  # Sort of assuming evenness now?
  grid=(mesh_len // 2,),
  in_specs=[
    # Our input reference is just our big tensor in HBM
    pl.BlockSpec(memory_space=pl.ANY)
  ],
  # Our output reference will be _another_ big tensor in HBM
  out_specs=pl.BlockSpec(memory_space=pl.ANY),
  # This will be an error if you need more semaphores for more neighbors
  scratch_shapes=(
    [pltpu.SemaphoreType.DMA] * 3 # local_copy_op, send_sem_left, send_sem_right
    + [pltpu.SemaphoreType.DMA] * 2 # These are our recv_sems (For 2x2, we only need 1 of them)
  )
)

  out_shape=jax.ShapeDtypeStruct((mesh_len * m, k), dtype=jnp.bfloat16)
  
  return pl.pallas_call(
    all_gather_kernel_bidi,
    grid_spec=grid_spec,
    out_shape=out_shape
  )(x)

In [33]:
xla_ag = jax.jit(
    jax.shard_map(
        lambda x: jax.lax.all_gather(x, 'x', tiled=True),
        mesh=mesh, in_specs=P('x', None), out_specs=P('x', None))
)(weights)

agbd = jax.jit(
  jax.shard_map(
      make_ag,
      mesh=mesh,
      in_specs=P('x', None),
      out_specs=P('x', None),
      check_vma=False
  )
)(weights)

jnp.allclose(xla_ag, agbd)

Array(True, dtype=bool)

In [ ]:
# NOTE: This is where we introduce semaphores as a way to mark data dependenices
# I.e., Results from one computation _depend_ on previous results**
# the semaphore lives on the device where wait() will be called**
# TODO: either yoink this from docs or write it

def all_reduce_bidi(
    input_ref, output_ref,
    send_sem_right, send_sem_left,
    recv_sem_right, recv_sem_left,
    vmem_scratch # Send this straight to VMEM
):

  pid = pl.program_id(0)
  shard_width = input_ref.shape[1]

  x_ring = jax.lax.axis_size('x')
  y_ring = jax.lax.axis_size('y')
  this_device_y = jax.lax.axis_index('y')
  this_device_x = jax.lax.axis_index('x')

  left_device_y = jax.lax.rem(this_device_y - 1 + y_ring, y_ring)
  right_device_y = jax.lax.rem(this_device_y + 1, y_ring)

  # Sending what we RECEIVED FROM OUR LEFT (my_y - pid)
  copy_slot_right = jax.lax.rem(this_device_y - pid + y_ring, y_ring)
  # Sending what we RECEIVED FROM OUR RIGHT (my_id + pid)
  copy_slot_left = jax.lax.rem(this_device_y + pid, y_ring)


  """
  - Send directly to remote VMEM
    - Overlap comms with local read latency
  - This is where you need to be careful about synchronizing**
    - This probably requires capacity semaphores
    - UNLESS you have enough space in VMEM to hold all of the data...
      - But even then you're just waiting to accumulate...
  - You can still pipeline this pretty aggressively with bidi
  - But this is also the copy/receiving concept**
    - Work from one slot, receive on another
  """

  right_dma = pltpu.make_async_remote_copy(
    src_ref=input_ref.at[:, pl.ds(copy_slot_right * shard_width)],
    dst_ref=vmem_scratch.at[:, pl.ds(copy_slot_right * shard_width)],
    send_sem=send_sem_right,
    recv_sem=recv_sem_right,
    device_id=(this_device_x, this_device_y + 1),
    device_id_type=pltpu.DeviceIdType.MESH
  )

  left_dma = pltpu.make_async_remote_copy(
    src_ref=input_ref.at[:, pl.ds(copy_slot_left * shard_width)],
    dst_ref=vmem_scratch.at[:, pl.ds(copy_slot_left * shard_width)],
    send_sem=send_sem_left,
    recv_sem=recv_sem_left,
    device_id=(this_device_x, this_device_y - 1),
    device_id_type=pltpu.DeviceIdType.MESH
  )

  right_dma.start()
  left_dma.start()

  my_data = input_ref[...]
  right_dma.wait()

def make_ar_bidi(x):
  grid_spec = pltpu.PrefetchScalarGridSpec(
      num_scalar_prefetch=0,
      # TODO: this is wrong beyond 2x2
      grid=(1,),
      in_specs=[
        pl.BlockSpec()
      ],
      out_specs=(
        jax.ShapeDtypeStruct((weights.shape), dtype=jnp.bfloat16),
      ),
      scratch_shapes=(
        [pltpu.SemaphoreType.DMA] * 2 # local sem, send left, send right
        + [pltpu.SemaphoreType.DMA] * 2 # recv right/left
        # This should be dynamic-ish
        # 2MB scratch for ping pong
        + [pltpu.VMEM((2, 1024, 1024), dtype=jnp.float32)]
      )
    )

  return pl.pallas_call(
      all_reduce_bidi,
      grid_spec=grid_spec,
      out_shape=out_shape,
      # compiler_params=pltpu.CompilerParams(
          # collective_id=0
      # )
  )(x)

In [ ]:
# Added granularity of halving the outgoing ICI copies (L/R)
# NOTE: SHRINKING THE STEP SIZES (data sent) CAN BETTER OVERLAP??
# Except ICI is slowest comms channel ; this is good to test experimentally**

In [34]:
def _emit_gemm(x_ref, w_ref, o_ref, *, bm, bk, bn):
    """
    Emit a tiled GEMM pipeline
    All refs are HBM. emit_pipeline handles VMEM tiling + double-buffering.
    """
    m, k_dim = x_ref.shape
    _, n = w_ref.shape
    grid = (m // bm, n // bn, k_dim // bk)

    def body(x_vmem, w_vmem, o_vmem, accum):
        @pl.when(pl.program_id(2) == 0)
        def _():
            accum[...] = jnp.zeros_like(accum)

        accum[...] += jnp.dot(
            x_vmem[...], w_vmem[...],
            preferred_element_type=jnp.float32,
        )

        @pl.when(pl.program_id(2) == pl.num_programs(2) - 1)
        def _():
            o_vmem[...] = accum[...].astype(o_vmem.dtype)

    @functools.partial(pl.run_scoped, accum=pltpu.VMEM((bm, bn), jnp.float32))
    def _(accum):
        pltpu.emit_pipeline(
            functools.partial(body, accum=accum),
            grid=grid,
            in_specs=[
                pl.BlockSpec((bm, bk), lambda i, j, k: (i, k)),
                pl.BlockSpec((bk, bn), lambda i, j, k: (k, j)),
            ],
            out_specs=pl.BlockSpec((bm, bn), lambda i, j, k: (i, j)),
            # should_accumulate_out=True
        )(x_ref, w_ref, o_ref)

In [35]:
"""
Instead of AG -> slice -> GEMM sequentially, overlap the weight
transfer with GEMM computation.

Each device (i, j) has:
  inputs[I_x, J_y]  (1024, 1024)
  weights[J_x, K]   (1024, 1024)

The correct partial product is: inputs[i,j] @ weights[j,:]
After AR over y: output[i,:] = sum_j inputs[i,j] @ weights[j,:]

For the AG over x (x_ring=2):
  x_idx == y_idx; local weights ARE weights[j,:], compute immediately
  x_idx != y_idx; need neighbor's weights, wait for RDMA

Start RDMA, run local GEMM (overlapping comm+compute on the devices that
already have the right chunk), then recompute with received weights only where needed.
"""

def fused_ag_gemm_kernel(
    input_ref,          # HBM: inputs shard (m_local, k_local)
    weight_ref,         # HBM: weights shard (k_local, n)
    output_ref,         # HBM: GEMM output (m_local, n)
    recv_weight_ref,    # HBM: workspace for received weights (k_local, n)
    send_sem, recv_sem, # RDMA semaphores
):
    """
    Fused AG + GEMM via collective permute.

    Outer kernel manages weight exchange (RDMA along x-ring).
    Inner pipelines handle the tiled matmul.

    On half the devices (x_idx == y_idx) the local weights are
    already correct, so GEMM runs entirely overlapped with RDMA.
    On the other half (x_idx != y_idx) we wait for the remote
    chunk and then recompute — still a win over a blocking AG
    because the first GEMM warmed the MXU pipeline.
    """
    x_idx = jax.lax.axis_index('x')
    y_idx = jax.lax.axis_index('y')
    x_ring = jax.lax.axis_size('x')
    right_neighbor = jax.lax.rem(x_idx + 1, x_ring)

    BM, BK, BN = 512, 1024, 1024

    # --- Step 1: Kick off async weight exchange along x-ring ---
    # Each device sends its shard right and receives from its left neighbor
    rdma = pltpu.make_async_remote_copy(
        src_ref=weight_ref,
        dst_ref=recv_weight_ref,
        send_sem=send_sem,
        recv_sem=recv_sem,
        device_id=(right_neighbor, y_idx),
        device_id_type=pltpu.DeviceIdType.MESH,
    )

    with jax.named_scope("Start weight exchange DMA"):
        rdma.start()

    # --- Step 2: Local GEMM (overlaps RDMA) ---
    # When x_idx == y_idx this IS the correct result.
    # When x_idx != y_idx this is wasted compute — but the MXU
    # stays busy while ICI transfers the weight shard we need.
    with jax.named_scope("On Diagonal GEMM"):
        @pl.when(x_idx == y_idx)
        def _on_diag_gemm():
            _emit_gemm(input_ref, weight_ref, output_ref, bm=BM, bk=BK, bn=BN)

    with jax.named_scope("Wait for comms"):
        rdma.wait()

    # --- Step 4: Recompute with correct weights where needed ---
    # NOTE: if @pl.when around emit_pipeline gives trouble, the
    # fallback is to always copy the correct chunk into
    # recv_weight_ref (local or received) and run one GEMM.
    # NOTE: THIS CAUSES A MASSIVE PAUSE ON 2 DEVICES IN THE PROFILE
    # We're using x_idx/y_idx to denote on diagonal devices, which we
    # have seen have the correct data to compute the products we need
    with jax.named_scope("Unhappy 2nd GEMM"):
        @pl.when(x_idx != y_idx)
        def _():
            _emit_gemm(input_ref, recv_weight_ref, output_ref, bm=BM, bk=BK, bn=BN)


def make_fused_ag_gemm(inputs, weights):
    m_local, k_local = inputs.shape
    _, n = weights.shape

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=(1,),
        in_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # inputs
            pl.BlockSpec(memory_space=pl.ANY),  # weights
        ],
        # Two outputs: real output + HBM workspace for received weights
        out_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # GEMM output
            pl.BlockSpec(memory_space=pl.ANY),  # recv weight buffer
        ],
        scratch_shapes=(
            [pltpu.SemaphoreType.DMA]  # send_sem
            + [pltpu.SemaphoreType.DMA] # recv_sem
        ),
    )

    out_shape = [
        jax.ShapeDtypeStruct((m_local, n), inputs.dtype),   # GEMM output
        jax.ShapeDtypeStruct((k_local, n), weights.dtype),  # recv workspace
    ]

    results = pl.pallas_call(
        fused_ag_gemm_kernel,
        grid_spec=grid_spec,
        out_shape=out_shape,
    )(inputs, weights)

    return results[0]  # discard the workspace

In [36]:
# PUT IT TOGETHER
# INTERLEAVE THE COMPUTE with ppermute
# This is where the fun begins... How many versions can we cook up?
# We want blocks that are multiples of (8, 128) -> Benefit from larger block sizes
# Largely because we have FOUR MXUs on TPUv5e -> Only 2 for TPUv6e, but they're 256x256
# TODO: These numbers are WAY out of wack

def fused_ag_gemm_ar(inputs, weights):
    partial = make_fused_ag_gemm(inputs, weights)
    return make_ar(partial)

fused_fn = jax.jit(jax.shard_map(
    fused_ag_gemm_ar,
    mesh=mesh,
    in_specs=(P('x', 'y'), P('x', None)),
    out_specs=P('x', None),
    check_vma=False,
))

fused_fn(inputs, weights)

Array([[92, -233, -7, ..., -17.75, 37, -173],
       [-75, 59, 85.5, ..., -80.5, 73, -66.5],
       [53, -19.25, 24.75, ..., -58.25, -175, 206],
       ...,
       [44.75, -113, 52, ..., -116.5, 150, 8.875],
       [-73, 284, -134, ..., -8, -376, 124.5],
       [-396, -89.5, -3.21875, ..., 92.5, -27.75, 11.25]], dtype=bfloat16)

In [37]:
test = fused_fn(inputs, weights)
numerics.compare(ref, test, atol=1e-2, rtol=1e-2, region_grid=(2, 2))

NumericsResult(FAIL)
  shape:     (16384, 8192)
  max_diff:  2.000000
  mean_diff: 0.000021
  median:    0.000000
  % > 0.1: 0.00%
  worst at (83, 2034): ref=-268.0000, test=-266.0000
  regions (2x2):
    [0,0]: mean=0.0000, %bad=0.0%
    [0,1]: mean=0.0000, %bad=0.0%
    [1,0]: mean=0.0000, %bad=0.0%
    [1,1]: mean=0.0000, %bad=0.0%

In [ ]:
"""
shard_map shows up as the SPMD region boundary in the device trace. Body has
two distinct XLA computations that can't fuse across the pallas_call custom call:

- make_fused_ag_gemm(...)
  - lowers to a pallas_call custom call.
- make_ar(...)
  - lowers to an all-reduce

Custom calls are fusion barriers, so XLA splits the shard_map body into
multiple partitioned computations
"""

In [38]:
# PERFORMANCE CHECK
benchmark(fused_fn, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:       12.493 ms
  median:     12.493 ms
  stdev:       0.008 ms
  min:        12.480 ms
  max:        12.512 ms
  p95:        12.507 ms
  p99:        12.511 ms

In [53]:
test_compiled = fused_fn.lower(inputs, weights).compile({'xla_enable_transpose_trace': True})
result = test_compiled(inputs, weights)
result.block_until_ready()

with jax.profiler.trace('./traces/fused_ag_gemm'):
    result = test_compiled(inputs, weights)
    result.block_until_ready()

In [ ]:
"""
This is where we assume that the all gather runs first
So we need to compute local, then send to neighboring
device for accumulation

Each device has full weights, each device needs to compute local GEMM
and pass the results to neighbor along y

We need to load correct weight tiles on each device, then do GEMM
Then send accumulated result to neighbor, then add, then materialize
"""

def _gemm_body_to_accum(x_vmem, w_vmem, *, accum):
    
    accum[...] += jnp.dot(
        x_vmem[...],
        w_vmem[...],
        preferred_element_type=jnp.float32,
    )


def _fused_gemm_ar_tile_kernel(
    x_ref, w_ref, out_ref,
    send_sem, recv_sem,
    send_buf, recv_buf,
    *,
    bm: int,
    bk: int,
    bn: int,
    k_local: int,
):
    x_idx = jax.lax.axis_index("x")
    y_idx = jax.lax.axis_index("y")
    y_ring = jax.lax.axis_size("y")
    right_y = jax.lax.rem(y_idx + 1, y_ring)

    @functools.partial(pl.run_scoped, accum=pltpu.VMEM((bm, bn), jnp.float32))
    def _scoped(accum):
        with jax.named_scope("init VMEM accum"):
            accum[...] = jnp.zeros_like(accum)

        pltpu.emit_pipeline(
            functools.partial(_gemm_body_to_accum, accum=accum),
            grid=(k_local // bk,),
            in_specs=[
                pl.BlockSpec((bm, bk), lambda kk: (0, kk)),
                pl.BlockSpec((bk, bn), lambda kk: (kk, 0)),
            ],
            # If we omit the out_specs arg, we need to remove this as well
            # out_specs=pl.BlockSpec((bm, bn), lambda kk: (0, 0)),
            should_accumulate_out=False,
        )(x_ref, w_ref)

        # Ring all-reduce over y in VMEM.
        with jax.named_scope("Write Accum to Send Buffer"):
            send_buf[...] = accum[...]

        for i in range(y_ring - 1):
            rdma = pltpu.make_async_remote_copy(
                src_ref=send_buf,
                dst_ref=recv_buf,
                send_sem=send_sem,
                recv_sem=recv_sem,
                device_id=(x_idx, right_y),
                device_id_type=pltpu.DeviceIdType.MESH,
            )
            with jax.named_scope(f"Initiate DMA {i}"):
                rdma.start()
            with jax.named_scope(f"Wait on DMA {i}"):
                rdma.wait()

            with jax.named_scope(f"Add received buffer to accumulator"):
                accum[...] += recv_buf[...]
            with jax.named_scope(f"Write receiver buffer into send buffer"):
                send_buf[...] = recv_buf[...]

        # Single final materialization to output tile.
        with jax.named_scope(f"Write results to HBM"):
            out_ref[...] = accum[...].astype(out_ref.dtype)

# NOTE: We're still not properly overlapping here
def make_fused_gemm_ar(
    x: jax.Array,
    w: jax.Array,
    *,
    bm: int = 512,
    bk: int = 512,
    bn: int = 512,
):
    m_local, k_local = x.shape
    k2, n = w.shape

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=(m_local // bm, n // bn),
        in_specs=[
            pl.BlockSpec((bm, k_local), lambda i, j: (i, j)),
            pl.BlockSpec((k_local, bn), lambda i, j: (i, j)),
        ],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j: (i, j)),
        scratch_shapes=(
            [pltpu.SemaphoreType.DMA] * 2
            + [pltpu.VMEM((bm, bn), jnp.float32)]  # send_buf
            + [pltpu.VMEM((bm, bn), jnp.float32)]  # recv_buf
        ),
    )

    out_shape = jax.ShapeDtypeStruct((m_local, n), x.dtype)
    return pl.pallas_call(
        functools.partial(
            _fused_gemm_ar_tile_kernel,
            bm=bm,
            bk=bk,
            bn=bn,
            k_local=k_local,
        ),
        grid_spec=grid_spec,
        out_shape=out_shape
    )(x, w)


def ag_then_fused_gemm_ar_kernel(inputs, weights):
    y_idx = jax.lax.axis_index("y")
    y_size = jax.lax.axis_size("y")

    w_full = make_ag(weights)
    k_full, n = w_full.shape
    k_block = k_full // y_size
    w_slice = jax.lax.dynamic_slice(w_full, (y_idx * k_block, 0), (k_block, n))

    return make_fused_gemm_ar(inputs, w_slice, bm=256, bk=512, bn=256)


fused_ag_then_gemm_ar = jax.jit(
    jax.shard_map(
        ag_then_fused_gemm_ar_kernel,
        mesh=mesh,
        in_specs=(P("x", "y"), P("x", None)),
        out_specs=P("x", None),
        check_vma=False,
    )
)

'\nThis is where we assume that the all gather runs first\nSo we need to compute local, then send to neighboring\ndevice for accumulation\n\nEach device has full weights, each device needs to compute local GEMM\nand pass the results to neighbor along y\n\nWe need to load correct weight tiles on each device, then do GEMM\nThen send accumulated result to neighbor, then add, then materialize\n'

In [ ]:
fused_fn2 = fused_ag_then_gemm_ar(inputs, weights)
numerics.compare(ref, test, atol=1e-2, rtol=1e-2, region_grid=(2, 2))

In [76]:
benchmark(fused_ag_then_gemm_ar, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:        4.063 ms
  median:      4.061 ms
  stdev:       0.008 ms
  min:         4.051 ms
  max:         4.089 ms
  p95:         4.074 ms
  p99:         4.085 ms

In [62]:
test_compiled = fused_ag_then_gemm_ar.lower(inputs, weights).compile({'xla_enable_transpose_trace': True})
result = test_compiled(inputs, weights)
result.block_until_ready()

with jax.profiler.trace('./traces/fused_ar_gemm'):
    result = test_compiled(inputs, weights)
    result.block_until_ready()

In [48]:
def _emit_gemm_2(x_ref, y_ref, *, accum):
    accum[...] += jnp.dot(
        x_ref[...],
        y_ref[...],
        preferred_element_type=jnp.float32,
    )

# TODO: _please_ rename the named calls, they are incoherent
# TODO: This impl OOMs on big shapes -- probably because we're grabbing EVERYTHING at once
# Need to actually tile over N in the block spec, switch things up here to read slices correctly
def _ag_overlapped_gemm_ar_kernel(
    x_ref, w_ref, out_ref,
    send_sem, recv_sem,
    send_buf, recv_buf,
    *,
    bm: int,
    bk: int,
    bn: int,
    n_tiles: int,
    k_local: int,
):
    this_x = jax.lax.axis_index("x")
    this_y = jax.lax.axis_index("y")
    y_ring = jax.lax.axis_size("y")
    right_y = jax.lax.rem(this_y + 1, y_ring)

    def _do_gemm(x, w, accum):
        accum[...] = jnp.zeros_like(accum)
        pltpu.emit_pipeline(
            functools.partial(_emit_gemm_2, accum=accum),
            grid=(k_local // bk,),
            in_specs=[
                pl.BlockSpec((bm, bk), lambda kk: (0, kk)),
                pl.BlockSpec((bk, bn), lambda kk: (kk, 0)),
            ],
            should_accumulate_out=False,
        )(x, w)

    @functools.partial(
        pl.run_scoped,
        accum_a=pltpu.VMEM((bm, bn), jnp.float32),
        accum_b=pltpu.VMEM((bm, bn), jnp.float32),
    )
    def _scoped(accum_a, accum_b):
        with jax.named_scope("Prologue GEMM"):
            # _do_gemm(x_ref, w_ref.at[:, pl.ds(0, bn)], accum_a)
            _do_gemm(x_ref, w_ref, accum_a, n_offset=0)

        for j in range(1, n_tiles):
            cur = accum_b if j % 2 == 1 else accum_a
            prev = accum_a if j % 2 == 1 else accum_b

            with jax.named_scope("Write prev buff to send buff"):
                send_buf[...] = prev[...]
            
            for i in range(y_ring - 1):
                rdma = pltpu.make_async_remote_copy(
                    src_ref=send_buf,
                    dst_ref=recv_buf,
                    send_sem=send_sem,
                    recv_sem=recv_sem,
                    device_id=(this_x, right_y),
                    device_id_type=pltpu.DeviceIdType.MESH,
                )
                rdma.start()
                if i == 0:
                    # _do_gemm(x_ref, w_ref[:, pl.ds(j * bn, bn)], cur)
                    with jax.named_scope("Prologue GEMM 2"):
                        _do_gemm(x_ref, w_ref, cur, n_offset=j)
                with jax.named_scope("Wait on Comms"):
                    rdma.wait()
                with jax.named_scope("Write recv to prev"):
                    prev[...] += recv_buf[...]
                with jax.named_scope("Swap recv and send"):
                    send_buf[...] = recv_buf[...]

            with jax.named_scope("write inner out buf"):
                out_ref[:, pl.ds((j - 1) * bn, bn)] = prev[...].astype(out_ref.dtype)

        last = accum_b if (n_tiles - 1) % 2 == 1 else accum_a
        if n_tiles == 1:
            last = accum_a

        send_buf[...] = last[...]
        for i in range(y_ring - 1):
            rdma = pltpu.make_async_remote_copy(
                src_ref=send_buf,
                dst_ref=recv_buf,
                send_sem=send_sem,
                recv_sem=recv_sem,
                device_id=(this_x, right_y),
                device_id_type=pltpu.DeviceIdType.MESH,
            )
            rdma.start()
            rdma.wait()
            last[...] += recv_buf[...]
            send_buf[...] = recv_buf[...]

        out_ref[:, pl.ds((n_tiles - 1) * bn, bn)] = last[...].astype(out_ref.dtype)


def make_ag_overlapped_gemm_ar(
    x: jax.Array,
    w: jax.Array,
    *,
    bm: int = 512,
    bk: int = 512,
    bn: int = 512,
):
    m_local, k_local = x.shape
    _, n = w.shape
    n_tiles = n // bn

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=(m_local // bm,),  # only tile over m
        in_specs=[
            pl.BlockSpec((bm, k_local), lambda i: (i, 0)),
            pl.BlockSpec((k_local, n), lambda i: (0, 0)),   # full n
        ],
        out_specs=pl.BlockSpec((bm, n), lambda i: (i, 0)),  # full n
        scratch_shapes=(
            pltpu.SemaphoreType.DMA,                # send_sem
            pltpu.SemaphoreType.DMA,                # recv_sem
            pltpu.VMEM((bm, bn), jnp.float32),     # send_buf
            pltpu.VMEM((bm, bn), jnp.float32),     # recv_buf
        ),
    )

    out_shape = jax.ShapeDtypeStruct((m_local, n), x.dtype)
    return pl.pallas_call(
        functools.partial(
            _ag_overlapped_gemm_ar_kernel,
            bm=bm,
            bk=bk,
            bn=bn,
            n_tiles=n_tiles,
            k_local=k_local,
        ),
        grid_spec=grid_spec,
        out_shape=out_shape,
    )(x, w)


def ag_then_overlapped_gemm_ar_kernel(inputs, weights):
    y_idx = jax.lax.axis_index("y")
    y_size = jax.lax.axis_size("y")

    w_full = make_ag(weights)
    k_full, n = w_full.shape
    k_block = k_full // y_size
    w_slice = jax.lax.dynamic_slice(w_full, (y_idx * k_block, 0), (k_block, n))

    return make_ag_overlapped_gemm_ar(inputs, w_slice)


fused_ag_then_overlapped_gemm_ar = jax.jit(
    jax.shard_map(
        ag_then_overlapped_gemm_ar_kernel,
        mesh=mesh,
        in_specs=(P("x", "y"), P("x", None)),
        out_specs=P("x", None),
        check_vma=False,
    )
)

In [ ]:
# Also OOMs because of weird usage of blockspecs here -> WAY too big
test_overlapped = fused_ag_then_overlapped_gemm_ar(inputs, weights)

In [ ]:
# When the shapes are 32k 32k 16k
"""
---------------------------------------------------------------------------
JaxRuntimeError                           Traceback (most recent call last)
Cell In[113], line 1
----> 1 test_overlapped = fused_ag_then_overlapped_gemm_ar(inputs, weights)

    [... skipping hidden 11 frame]

File ~/sfp/.venv/lib/python3.13/site-packages/jax/_src/compiler.py:362, in backend_compile_and_load(backend, module, executable_devices, options, host_callbacks)
    353       return backend.compile_and_load(
    354           module,
    355           executable_devices=executable_devices,
    356           compile_options=options,
    357           host_callbacks=host_callbacks,
    358       )
    359     # Some backends don't have `host_callbacks` option yet
    360     # TODO(sharadmv): remove this fallback when all backends allow `compile`
    361     # to take in `host_callbacks`
--> 362     return backend.compile_and_load(
    363         module,
    364         executable_devices=executable_devices,
    365         compile_options=options,
    366     )
    367 except _jax.JaxRuntimeError as e:
    368   for error_handler in _XLA_RUNTIME_ERROR_HANDLERS:

JaxRuntimeError: RESOURCE_EXHAUSTED: Allocation (size=536870912) would exceed memory (size=134217728) :: #allocation16 [shape = 'u8[536870912]{0}', space=vmem, size = 0x20000000, tag = 'input window allocation for operator input 1. The window shape is bf16[16384,16384], while the full shape is bf16[16384,16384]. The u8 shape/size shown above represents the raw byte size of this window in VMEM. This allocation is single buffered.'] :: shard_map.20
"""

In [79]:
test_overlapped

Array([[112, -48, -129, ..., 29.375, 28.125, 197],
       [-102.5, 37.5, 89, ..., -26.5, -73.5, 73.5],
       [139, -69, 96, ..., -200, 115.5, -125],
       ...,
       [-10.875, 33, 45.25, ..., -127.5, -130, 8.5625],
       [-15.875, -212, -89, ..., -42.25, 76.5, -139],
       [-3.9375, -102, 65.5, ..., 19.75, 41, 5.0625]], dtype=bfloat16)

In [80]:
numerics.compare(ref, test_overlapped, atol=1.0, rtol=1e-2, region_grid=(2,2))

NumericsResult(PASS)
  shape:     (8192, 4096)
  max_diff:  2.000000
  mean_diff: 0.119141
  median:    0.000000
  % > 0.1: 30.56%
  worst at (0, 366): ref=292.0000, test=290.0000
  regions (2x2):
    [0,0]: mean=0.1191, %bad=30.6%
    [0,1]: mean=0.1191, %bad=30.5%
    [1,0]: mean=0.1191, %bad=30.6%
    [1,1]: mean=0.1191, %bad=30.5%

In [81]:
benchmark(fused_ag_then_overlapped_gemm_ar, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:        2.685 ms
  median:      2.685 ms
  stdev:       0.010 ms
  min:         2.665 ms
  max:         2.712 ms
  p95:         2.703 ms
  p99:         2.710 ms

In [82]:
test_compiled_overlapped = fused_ag_then_overlapped_gemm_ar.lower(inputs, weights).compile({'xla_enable_transpose_trace': True})
result = test_compiled_overlapped(inputs, weights)
result.block_until_ready()

with jax.profiler.trace('./traces/overlapped_ar_gemm'):
    result = test_compiled_overlapped(inputs, weights)
    result.block_until_ready()

In [60]:
def _emit_gemm_vmem(x_ref, w_ref, *, bm, bk, bn):
    """
    Need to make sure that the results of this accumulator get stashed somewhere
    """
    m, k_dim = x_ref.shape
    _, n = w_ref.shape
    grid = (m // bm, n // bn, k_dim // bk)

    def body(x_vmem, w_vmem, accum):
        @pl.when(pl.program_id(2) == 0)
        def _():
            accum[...] = jnp.zeros_like(accum)

        accum[...] += jnp.dot(
            x_vmem[...], w_vmem[...],
            preferred_element_type=jnp.float32,
        )

    @functools.partial(pl.run_scoped, accum=pltpu.VMEM((bm, bn), jnp.float32))
    def _(accum):
        pltpu.emit_pipeline(
            functools.partial(body, accum=accum),
            grid=grid,
            in_specs=[
                pl.BlockSpec((bm, bk), lambda i, j, k: (i, k)),
                pl.BlockSpec((bk, bn), lambda i, j, k: (k, j)),
            ],
            out_specs=pl.BlockSpec((bm, bn), lambda i, j, k: (i, j)),
            # should_accumulate_out=True
        )(x_ref, w_ref)

In [ ]:
"""
What we really need to do is think about:
  - How to fully fuse everything
  - How to scale it to larger topologies
    - 2x2 is pretty boring from a comms perspective
    - Doesn't really enable any bidi stuff
    - Also doesn't show the warts w.r.t run-ahead, etc.
"""

def fused_pperm_reduce_kernel_1D(
    input_ref, weight_ref, hbm_scratch, output_ref,
    right_send_sem, right_recv_sem, left_send_sem, left_recv_sem,
    recv_bufer, send_buffer

):
    x_idx = jax.lax.axis_index('x')
    y_idx = jax.lax.axis_index('y')
    x_size = jax.lax.axis_size('x')
    y_size = jax.lax.axis_size('y')

    x_neighbor = jax.lax.rem(x_idx + 1, x_size)
    y_neighbor = jax.lax.rem(y_idx + 1, y_size)

    # Need to zero our outs/accums

    # Over our x axis -- sending the weights needed for compute
    x_dma = pltpu.make_async_remote_copy(
        src_ref=weight_ref,
        dst_ref=hbm_scratch,
        send_sem=,
        recv_sem=,
        device_id=(x_neighbor, y_idx),
        device_id_type=pltpu.DeviceIdType.MESH
    )

    # Over the y axis -- sending our reduction results
    y_dma = pltpu.make_async_remote_copy(
        src_ref=,
        dst_ref=,
        send_sem=,
        recv_sem=,
        device_id=(x_idx, y_neighbor),
        device_id_type=pltpu.DeviceIdType.MESH
    )


def make_fully_fused_1D(
    x: jax.Array,
    y: jax.Array,
    bm,
    bk,
    bn
):
    m, k = x.shape
    _, n = y.shape

    # This is an awkward pattern --
    # Have to keep the outer kernel grid in line with the matmul pipeline**
    tm, tk, tn = m // bm, k // bk, n // bn

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=(1,),
        in_specs=[
            pl.BlockSpec(memory_space=pl.ANY), # LHS just a chunk of memory in HBM
            pl.BlockSpec(memory_space=pl.ANY) # RHS just a chunk of memory in HBM
        ],
        out_specs=[
            pl.BlockSpec(memory_space=pl.ANY), # hbm_scratch
            pl.BlockSpec(memory_space=pl.ANY) # output_ref
        ],
        scratch_shapes=(
            pltpu.SemaphoreType.DMA,
            pltpu.SemaphoreType.DMA,
            pltpu.SemaphoreType.DMA,
            pltpu.SemaphoreType.DMA,
            pltpu.VMEM((tm, tn)), # recv buffer
            pltpu.VMEM((tm, tn)) # send buffer
        )
    )

    out_shape=jax.ShapeDtypeStruct((y.shape), dtype=y.dtype)

    return pl.pallas_call(
        fused_pperm_reduce_kernel_1D,
        grid_spec=grid_spec,
        out_shape=out_shape,
    )

def make_fused_pperm(
    x: jax.Array,
    y: jax.Array,
):
    pass

In [56]:
def _gemm_to_accum(x_vmem, w_vmem, *, accum):
    """emit_pipeline body: MAC into a scoped VMEM accumulator."""
    accum[...] += jnp.dot(
        x_vmem[...], w_vmem[...],
        preferred_element_type=jnp.float32,
    )


def _tiled_gemm(x_ref, w_ref, accum, *, bm, bk, bn):
    """
    Run a single tiled GEMM: x_ref @ w_ref -> accum.
    x_ref and w_ref are HBM refs; accum is a scoped VMEM ref (bm, bn) f32.
    emit_pipeline handles all VMEM staging.
    """
    k_dim = x_ref.shape[1]
    accum[...] = jnp.zeros_like(accum)
    pltpu.emit_pipeline(
        functools.partial(_gemm_to_accum, accum=accum),
        grid=(k_dim // bk,),
        in_specs=[
            pl.BlockSpec((bm, bk), lambda kk: (0, kk)),
            pl.BlockSpec((bk, bn), lambda kk: (kk, 0)),
        ],
        should_accumulate_out=False,
    )(x_ref, w_ref)


def _ar_ring_reduce(accum, send_buf, recv_buf, send_sem, recv_sem,
                    *, x_idx, right_y, y_ring):
    """
    In-place ring all-reduce of accum along the y-axis.
    All buffers are VMEM. Blocking ring (no pipelining of hops).
    """
    send_buf[...] = accum[...]
    for _ in range(y_ring - 1):
        rdma = pltpu.make_async_remote_copy(
            src_ref=send_buf,
            dst_ref=recv_buf,
            send_sem=send_sem,
            recv_sem=recv_sem,
            device_id=(x_idx, right_y),
            device_id_type=pltpu.DeviceIdType.MESH,
        )
        rdma.start()
        rdma.wait()
        accum[...] += recv_buf[...]
        send_buf[...] = recv_buf[...]


def _fused_ag_gemm_ar_kernel(
    x_ref,              # HBM (m_local, k_local) bf16
    w_ref,              # HBM (k_local, n) bf16 — local weight shard
    out_ref,            # HBM (m_local, n) bf16
    recv_w_ref,         # HBM (k_local, n) bf16 — weight recv workspace
    ag_send_sem, ag_recv_sem,   # DMA sems for AG along x
    ar_send_sem, ar_recv_sem,   # DMA sems for AR along y
    send_buf, recv_buf,         # VMEM (bm, bn) f32 — AR comm buffers
    *,
    bm: int,
    bk: int,
    bn: int,
):
    x_idx = jax.lax.axis_index("x")
    y_idx = jax.lax.axis_index("y")
    x_ring = jax.lax.axis_size("x")
    y_ring = jax.lax.axis_size("y")
    right_x = jax.lax.rem(x_idx + 1, x_ring)
    right_y = jax.lax.rem(y_idx + 1, y_ring)

    m_local = x_ref.shape[0]
    _, n = w_ref.shape
    n_tiles = n // bn
    m_tiles = m_local // bm

    @functools.partial(
        pl.run_scoped,
        accum_a=pltpu.VMEM((bm, bn), jnp.float32),
        accum_b=pltpu.VMEM((bm, bn), jnp.float32),
    )
    def _body(accum_a, accum_b):
        for mi in range(m_tiles):
            # Slice the m-tile of x once (HBM view)
            x_tile = x_ref.at[pl.ds(mi * bm, bm), :]

            # ── AG rotate + GEMM for tile (mi, 0): PROLOGUE ──
            # Rotate weight shards along x, GEMM fires when shard matches y_idx
            for step in range(x_ring):
                shard_idx = jax.lax.rem(x_idx + x_ring - step, x_ring)
                current_w = w_ref if step == 0 else recv_w_ref

                if step < x_ring - 1:
                    src = w_ref if step == 0 else recv_w_ref
                    ag_rdma = pltpu.make_async_remote_copy(
                        src_ref=src,
                        dst_ref=recv_w_ref,
                        send_sem=ag_send_sem,
                        recv_sem=ag_recv_sem,
                        device_id=(right_x, y_idx),
                        device_id_type=pltpu.DeviceIdType.MESH,
                    )
                    ag_rdma.start()

                @pl.when(shard_idx == y_idx)
                def _():
                    w_tile = current_w.at[:, pl.ds(0, bn)]
                    _tiled_gemm(x_tile, w_tile, accum_a, bm=bm, bk=bk, bn=bn)

                if step < x_ring - 1:
                    ag_rdma.wait()

            # ── STEADY STATE: AR(j-1) overlapped with AG+GEMM(j) ──
            for j in range(1, n_tiles):
                cur = accum_b if j % 2 == 1 else accum_a
                prev = accum_a if j % 2 == 1 else accum_b

                # Start AR for prev tile
                send_buf[...] = prev[...]

                for hop in range(y_ring - 1):
                    ar_rdma = pltpu.make_async_remote_copy(
                        src_ref=send_buf,
                        dst_ref=recv_buf,
                        send_sem=ar_send_sem,
                        recv_sem=ar_recv_sem,
                        device_id=(x_idx, right_y),
                        device_id_type=pltpu.DeviceIdType.MESH,
                    )
                    ar_rdma.start()

                    # Overlap: on the first AR hop, run the next GEMM
                    if hop == 0:
                        for step in range(x_ring):
                            shard_idx = jax.lax.rem(
                                x_idx + x_ring - step, x_ring
                            )
                            current_w = w_ref if step == 0 else recv_w_ref

                            if step < x_ring - 1:
                                src = w_ref if step == 0 else recv_w_ref
                                ag_rdma = pltpu.make_async_remote_copy(
                                    src_ref=src,
                                    dst_ref=recv_w_ref,
                                    send_sem=ag_send_sem,
                                    recv_sem=ag_recv_sem,
                                    device_id=(right_x, y_idx),
                                    device_id_type=pltpu.DeviceIdType.MESH,
                                )
                                ag_rdma.start()

                            @pl.when(shard_idx == y_idx)
                            def _():
                                w_tile = current_w.at[:, pl.ds(j * bn, bn)]
                                _tiled_gemm(
                                    x_tile, w_tile, cur,
                                    bm=bm, bk=bk, bn=bn,
                                )

                            if step < x_ring - 1:
                                ag_rdma.wait()

                    ar_rdma.wait()
                    prev[...] += recv_buf[...]
                    send_buf[...] = recv_buf[...]

                # Write fully-reduced prev tile to HBM
                out_ref.at[
                    pl.ds(mi * bm, bm), pl.ds((j - 1) * bn, bn)
                ].set(prev[...].astype(out_ref.dtype))

            # ── EPILOGUE: drain AR for last tile ──
            last = accum_b if (n_tiles - 1) % 2 == 1 else accum_a
            if n_tiles == 1:
                last = accum_a

            _ar_ring_reduce(
                last, send_buf, recv_buf, ar_send_sem, ar_recv_sem,
                x_idx=x_idx, right_y=right_y, y_ring=y_ring,
            )
            out_ref.at[
                pl.ds(mi * bm, bm), pl.ds((n_tiles - 1) * bn, bn)
            ].set(last[...].astype(out_ref.dtype))


def make_fused_ag_gemm_ar(
    x: jax.Array,
    w: jax.Array,
    *,
    bm: int = 512,
    bk: int = 512,
    bn: int = 512,
):
    m_local, k_local = x.shape
    _, n = w.shape

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=(1,),
        in_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # x
            pl.BlockSpec(memory_space=pl.ANY),  # w
        ],
        out_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # output
            pl.BlockSpec(memory_space=pl.ANY),  # recv weight workspace
        ],
        scratch_shapes=(
            pltpu.SemaphoreType.DMA,            # ag_send_sem
            pltpu.SemaphoreType.DMA,            # ag_recv_sem
            pltpu.SemaphoreType.DMA,            # ar_send_sem
            pltpu.SemaphoreType.DMA,            # ar_recv_sem
            pltpu.VMEM((bm, bn), jnp.float32),  # send_buf
            pltpu.VMEM((bm, bn), jnp.float32),  # recv_buf
        ),
    )

    out_shape = [
        jax.ShapeDtypeStruct((m_local, n), x.dtype),
        jax.ShapeDtypeStruct((k_local, n), w.dtype),
    ]

    results = pl.pallas_call(
        functools.partial(
            _fused_ag_gemm_ar_kernel,
            bm=bm, bk=bk, bn=bn,
        ),
        grid_spec=grid_spec,
        out_shape=out_shape,
    )(x, w)

    return results[0]


def fused_ag_gemm_ar_kernel(inputs, weights):
    return make_fused_ag_gemm_ar(inputs, weights)


fused_ag_gemm_ar_fn = jax.jit(
    jax.shard_map(
        fused_ag_gemm_ar_kernel,
        mesh=mesh,
        in_specs=(P("x", "y"), P("x", None)),
        out_specs=P("x", None),
        check_vma=False,
    )
)

In [ ]:
fused_ag_gemm_ar_fn(inputs, weights)

In [ ]:
def _gemm_to_accum(x_vmem, w_vmem, *, accum):
    accum[...] += jnp.dot(
        x_vmem[...], w_vmem[...],
        preferred_element_type=jnp.float32,
    )


def _tiled_gemm(x_ref, w_ref, accum, *, bm, bk, bn):
    """
    Run a single tiled GEMM: x_ref @ w_ref -> accum.
    x_ref and w_ref are HBM refs; accum is a scoped VMEM ref (bm, bn) f32.
    emit_pipeline handles all VMEM staging.
    """
    k_dim = x_ref.shape[1]
    accum[...] = jnp.zeros_like(accum)
    pltpu.emit_pipeline(
        functools.partial(_gemm_to_accum, accum=accum),
        grid=(k_dim // bk,),
        in_specs=[
            pl.BlockSpec((bm, bk), lambda kk: (0, kk)),
            pl.BlockSpec((bk, bn), lambda kk: (kk, 0)),
        ],
        should_accumulate_out=False,
    )(x_ref, w_ref)


def _wb_to_hbm(src_vmem, dst_hbm_slice, staging, sem):
    """
    Write a VMEM f32 accumulator tile to an HBM bf16 output ref.
    Cast into a bf16 staging buffer, then DMA to HBM.
    """
    staging[...] = src_vmem[...].astype(staging.dtype)
    wb = pltpu.make_async_copy(
        src_ref=staging,
        dst_ref=dst_hbm_slice,
        sem=sem,
    )
    wb.start()
    wb.wait()


def _ar_ring_reduce(accum, send_buf, recv_buf, send_sem, recv_sem,
                    *, x_idx, right_y, y_ring):
    """
    In-place ring all-reduce of accum along the y-axis.
    All buffers are VMEM. Blocking ring (no pipelining of hops).
    """
    send_buf[...] = accum[...]
    for _ in range(y_ring - 1):
        rdma = pltpu.make_async_remote_copy(
            src_ref=send_buf,
            dst_ref=recv_buf,
            send_sem=send_sem,
            recv_sem=recv_sem,
            device_id=(x_idx, right_y),
            device_id_type=pltpu.DeviceIdType.MESH,
        )
        rdma.start()
        rdma.wait()
        accum[...] += recv_buf[...]
        send_buf[...] = recv_buf[...]


def _fused_ag_gemm_ar_kernel(
    x_ref,              # HBM (m_local, k_local) bf16
    w_ref,              # HBM (k_local, n) bf16 — local weight shard
    out_ref,            # HBM (m_local, n) bf16
    recv_w_ref,         # HBM (k_local, n) bf16 — weight recv workspace
    ag_send_sem, ag_recv_sem,   # DMA sems for AG along x
    ar_send_sem, ar_recv_sem,   # DMA sems for AR along y
    wb_sem,                     # DMA sem for VMEM→HBM writeback
    send_buf, recv_buf,         # VMEM (bm, bn) f32 — AR comm buffers
    out_staging,                # VMEM (bm, bn) bf16 — HBM writeback staging
    *,
    bm: int,
    bk: int,
    bn: int,
):
    x_idx = jax.lax.axis_index("x")
    y_idx = jax.lax.axis_index("y")
    x_ring = jax.lax.axis_size("x")
    y_ring = jax.lax.axis_size("y")
    right_x = jax.lax.rem(x_idx + 1, x_ring)
    right_y = jax.lax.rem(y_idx + 1, y_ring)

    m_local = x_ref.shape[0]
    _, n = w_ref.shape
    n_tiles = n // bn
    m_tiles = m_local // bm

    @functools.partial(
        pl.run_scoped,
        accum_a=pltpu.VMEM((bm, bn), jnp.float32),
        accum_b=pltpu.VMEM((bm, bn), jnp.float32),
    )
    def _body(accum_a, accum_b):
        for mi in range(m_tiles):
            # Slice the m-tile of x once (HBM view)
            x_tile = x_ref.at[pl.ds(mi * bm, bm), :]

            # ── AG rotate + GEMM for tile (mi, 0): PROLOGUE ──
            # Rotate weight shards along x, GEMM fires when shard matches y_idx
            for step in range(x_ring):
                shard_idx = jax.lax.rem(x_idx + x_ring - step, x_ring)
                current_w = w_ref if step == 0 else recv_w_ref

                if step < x_ring - 1:
                    src = w_ref if step == 0 else recv_w_ref
                    ag_rdma = pltpu.make_async_remote_copy(
                        src_ref=src,
                        dst_ref=recv_w_ref,
                        send_sem=ag_send_sem,
                        recv_sem=ag_recv_sem,
                        device_id=(right_x, y_idx),
                        device_id_type=pltpu.DeviceIdType.MESH,
                    )
                    ag_rdma.start()

                @pl.when(shard_idx == y_idx)
                def _():
                    w_tile = current_w.at[:, pl.ds(0, bn)]
                    _tiled_gemm(x_tile, w_tile, accum_a, bm=bm, bk=bk, bn=bn)

                if step < x_ring - 1:
                    ag_rdma.wait()

            # ── STEADY STATE: AR(j-1) overlapped with AG+GEMM(j) ──
            for j in range(1, n_tiles):
                cur = accum_b if j % 2 == 1 else accum_a
                prev = accum_a if j % 2 == 1 else accum_b

                # Start AR for prev tile
                send_buf[...] = prev[...]

                for hop in range(y_ring - 1):
                    ar_rdma = pltpu.make_async_remote_copy(
                        src_ref=send_buf,
                        dst_ref=recv_buf,
                        send_sem=ar_send_sem,
                        recv_sem=ar_recv_sem,
                        device_id=(x_idx, right_y),
                        device_id_type=pltpu.DeviceIdType.MESH,
                    )
                    ar_rdma.start()

                    # Overlap: on the first AR hop, run the next GEMM
                    if hop == 0:
                        for step in range(x_ring):
                            shard_idx = jax.lax.rem(
                                x_idx + x_ring - step, x_ring
                            )
                            current_w = w_ref if step == 0 else recv_w_ref

                            if step < x_ring - 1:
                                src = w_ref if step == 0 else recv_w_ref
                                ag_rdma = pltpu.make_async_remote_copy(
                                    src_ref=src,
                                    dst_ref=recv_w_ref,
                                    send_sem=ag_send_sem,
                                    recv_sem=ag_recv_sem,
                                    device_id=(right_x, y_idx),
                                    device_id_type=pltpu.DeviceIdType.MESH,
                                )
                                ag_rdma.start()

                            @pl.when(shard_idx == y_idx)
                            def _():
                                w_tile = current_w.at[:, pl.ds(j * bn, bn)]
                                _tiled_gemm(
                                    x_tile, w_tile, cur,
                                    bm=bm, bk=bk, bn=bn,
                                )

                            if step < x_ring - 1:
                                ag_rdma.wait()

                    ar_rdma.wait()
                    prev[...] += recv_buf[...]
                    send_buf[...] = recv_buf[...]

                # Write fully-reduced prev tile to HBM via DMA
                _wb_to_hbm(
                    prev,
                    out_ref.at[pl.ds(mi * bm, bm), pl.ds((j - 1) * bn, bn)],
                    out_staging, wb_sem,
                )

            # ── EPILOGUE: drain AR for last tile ──
            last = accum_b if (n_tiles - 1) % 2 == 1 else accum_a
            if n_tiles == 1:
                last = accum_a

            _ar_ring_reduce(
                last, send_buf, recv_buf, ar_send_sem, ar_recv_sem,
                x_idx=x_idx, right_y=right_y, y_ring=y_ring,
            )
            _wb_to_hbm(
                last,
                out_ref.at[pl.ds(mi * bm, bm), pl.ds((n_tiles - 1) * bn, bn)],
                out_staging, wb_sem,
            )


def make_fused_ag_gemm_ar(
    x: jax.Array,
    w: jax.Array,
    *,
    bm: int = 512,
    bk: int = 512,
    bn: int = 512,
):
    m_local, k_local = x.shape
    _, n = w.shape

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=(1,),
        in_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # x
            pl.BlockSpec(memory_space=pl.ANY),  # w
        ],
        out_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # output
            pl.BlockSpec(memory_space=pl.ANY),  # recv weight workspace
        ],
        scratch_shapes=(
            pltpu.SemaphoreType.DMA,             # ag_send_sem
            pltpu.SemaphoreType.DMA,             # ag_recv_sem
            pltpu.SemaphoreType.DMA,             # ar_send_sem
            pltpu.SemaphoreType.DMA,             # ar_recv_sem
            pltpu.SemaphoreType.DMA,             # wb_sem
            pltpu.VMEM((bm, bn), jnp.float32),  # send_buf
            pltpu.VMEM((bm, bn), jnp.float32),  # recv_buf
            pltpu.VMEM((bm, bn), jnp.bfloat16), # out_staging
        ),
    )

    out_shape = [
        jax.ShapeDtypeStruct((m_local, n), x.dtype),
        jax.ShapeDtypeStruct((k_local, n), w.dtype),
    ]

    results = pl.pallas_call(
        functools.partial(
            _fused_ag_gemm_ar_kernel,
            bm=bm, bk=bk, bn=bn,
        ),
        grid_spec=grid_spec,
        out_shape=out_shape,
    )(x, w)

    return results[0]


def fused_ag_gemm_ar_kernel(inputs, weights):
    return make_fused_ag_gemm_ar(inputs, weights)


fused_ag_gemm_ar_fn = jax.jit(
    jax.shard_map(
        fused_ag_gemm_ar_kernel,
        mesh=mesh,
        in_specs=(P("x", "y"), P("x", None)),
        out_specs=P("x", None),
        check_vma=False,
    )
)

In [ ]:
fused_ag_gemm_ar_fn(inputs, weights)

Array([[91.5, -233, -7.09375, ..., -17.625, 36.75, -173],
       [-75.5, 58.5, 85.5, ..., -80.5, 73, -66.5],
       [53, -19.5, 24.75, ..., -58.25, -175, 206],
       ...,
       [-96, -33.25, 334, ..., -116.5, 150, 8.875],
       [-9.625, 68, -105.5, ..., -7.90625, -376, 124.5],
       [-152, -24.625, 9.25, ..., 92.5, -27.75, 11.1875]], dtype=bfloat16)

In [61]:
benchmark(fused_ag_gemm_ar_fn, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:      751.162 ms
  median:    751.157 ms
  stdev:       0.025 ms
  min:       751.128 ms
  max:       751.267 ms
  p95:       751.202 ms
  p99:       751.245 ms

In [ ]:
ff = fused_ag_gemm_ar_fn(inputs, weights)
ff.block_until_ready()

with jax.profiler.trace('./traces/fully_fused'):
    ff = fused_ag_gemm_ar_fn(inputs, weights)
    ff.block_until_ready()

In [64]:
def _gemm_to_accum(x_vmem, w_vmem, *, accum):
    """emit_pipeline body: MAC into a scoped VMEM accumulator."""
    accum[...] += jnp.dot(
        x_vmem[...], w_vmem[...],
        preferred_element_type=jnp.float32,
    )


def _tiled_gemm(x_ref, w_ref, accum, *, bm, bk, bn):
    """
    Run a single tiled GEMM: x_ref @ w_ref -> accum.
    x_ref and w_ref are HBM refs; accum is a scoped VMEM ref (bm, bn) f32.
    emit_pipeline handles all VMEM staging.
    """
    k_dim = x_ref.shape[1]
    accum[...] = jnp.zeros_like(accum)
    pltpu.emit_pipeline(
        functools.partial(_gemm_to_accum, accum=accum),
        grid=(k_dim // bk,),
        in_specs=[
            pl.BlockSpec((bm, bk), lambda kk: (0, kk)),
            pl.BlockSpec((bk, bn), lambda kk: (kk, 0)),
        ],
        should_accumulate_out=False,
    )(x_ref, w_ref)


def _wb_to_hbm(src_vmem, dst_hbm_slice, staging, sem):
    """
    Write a VMEM f32 accumulator tile to an HBM bf16 output ref.
    Cast into a bf16 staging buffer, then DMA to HBM.
    """
    staging[...] = src_vmem[...].astype(staging.dtype)
    wb = pltpu.make_async_copy(
        src_ref=staging,
        dst_ref=dst_hbm_slice,
        sem=sem,
    )
    wb.start()
    wb.wait()


def _ar_ring_reduce(accum, send_buf, recv_buf, send_sem, recv_sem,
                    *, x_idx, right_y, y_ring):
    """
    In-place ring all-reduce of accum along the y-axis.
    All buffers are VMEM. Blocking ring (no pipelining of hops).
    """
    send_buf[...] = accum[...]
    for _ in range(y_ring - 1):
        rdma = pltpu.make_async_remote_copy(
            src_ref=send_buf,
            dst_ref=recv_buf,
            send_sem=send_sem,
            recv_sem=recv_sem,
            device_id=(x_idx, right_y),
            device_id_type=pltpu.DeviceIdType.MESH,
        )
        rdma.start()
        rdma.wait()
        accum[...] += recv_buf[...]
        send_buf[...] = recv_buf[...]


def _fused_ag_gemm_ar_kernel(
    x_ref,              # HBM (m_local, k_local) bf16
    w_ref,              # HBM (k_local, n) bf16 — local weight shard
    out_ref,            # HBM (m_local, n) bf16
    recv_w_ref,         # HBM (k_local, n) bf16 — weight recv workspace
    stashed_w_ref,      # HBM (k_local, n) bf16 — correct shard after AG
    ag_send_sem, ag_recv_sem,   # DMA sems for AG along x
    ar_send_sem, ar_recv_sem,   # DMA sems for AR along y
    wb_sem,                     # DMA sem for VMEM→HBM writeback
    send_buf, recv_buf,         # VMEM (bm, bn) f32 — AR comm buffers
    out_staging,                # VMEM (bm, bn) bf16 — HBM writeback staging
    *,
    bm: int,
    bk: int,
    bn: int,
):
    x_idx = jax.lax.axis_index("x")
    y_idx = jax.lax.axis_index("y")
    x_ring = jax.lax.axis_size("x")
    y_ring = jax.lax.axis_size("y")
    right_x = jax.lax.rem(x_idx + 1, x_ring)
    right_y = jax.lax.rem(y_idx + 1, y_ring)

    m_local = x_ref.shape[0]
    _, n = w_ref.shape
    n_tiles = n // bn
    m_tiles = m_local // bm

    @functools.partial(
        pl.run_scoped,
        accum_a=pltpu.VMEM((bm, bn), jnp.float32),
        accum_b=pltpu.VMEM((bm, bn), jnp.float32),
    )
    def _body(accum_a, accum_b):
        for mi in range(m_tiles):
            x_tile = x_ref.at[pl.ds(mi * bm, bm), :]

            # ── AG: rotate weight shards once, stash the correct one ──
            # After this loop, stashed_w_ref has the shard where
            # shard_idx == y_idx. We reuse it for ALL n-tiles.
            for step in range(x_ring):
                shard_idx = jax.lax.rem(x_idx + x_ring - step, x_ring)
                current_w = w_ref if step == 0 else recv_w_ref

                if step < x_ring - 1:
                    src = w_ref if step == 0 else recv_w_ref
                    ag_rdma = pltpu.make_async_remote_copy(
                        src_ref=src,
                        dst_ref=recv_w_ref,
                        send_sem=ag_send_sem,
                        recv_sem=ag_recv_sem,
                        device_id=(right_x, y_idx),
                        device_id_type=pltpu.DeviceIdType.MESH,
                    )
                    ag_rdma.start()

                # When the matching shard arrives, stash it so later
                # ring steps don't clobber it.
                @pl.when(shard_idx == y_idx)
                def _():
                    stash = pltpu.make_async_copy(
                        src_ref=current_w,
                        dst_ref=stashed_w_ref,
                        sem=wb_sem,
                    )
                    stash.start()
                    stash.wait()

                if step < x_ring - 1:
                    ag_rdma.wait()

            # stashed_w_ref now holds the correct (k_local, n) shard.
            # All n-tile GEMMs read column slices from it — no more AG.

            # ── PROLOGUE: GEMM for n-tile 0 ──
            _tiled_gemm(
                x_tile, stashed_w_ref.at[:, pl.ds(0, bn)],
                accum_a, bm=bm, bk=bk, bn=bn,
            )

            # ── STEADY STATE: AR(j-1) overlapped with GEMM(j) ──
            for j in range(1, n_tiles):
                cur = accum_b if j % 2 == 1 else accum_a
                prev = accum_a if j % 2 == 1 else accum_b

                send_buf[...] = prev[...]

                for hop in range(y_ring - 1):
                    ar_rdma = pltpu.make_async_remote_copy(
                        src_ref=send_buf,
                        dst_ref=recv_buf,
                        send_sem=ar_send_sem,
                        recv_sem=ar_recv_sem,
                        device_id=(x_idx, right_y),
                        device_id_type=pltpu.DeviceIdType.MESH,
                    )
                    ar_rdma.start()

                    # Overlap: GEMM for tile j runs during first AR hop
                    if hop == 0:
                        _tiled_gemm(
                            x_tile,
                            stashed_w_ref.at[:, pl.ds(j * bn, bn)],
                            cur, bm=bm, bk=bk, bn=bn,
                        )

                    ar_rdma.wait()
                    prev[...] += recv_buf[...]
                    send_buf[...] = recv_buf[...]

                # Write fully-reduced prev tile to HBM
                _wb_to_hbm(
                    prev,
                    out_ref.at[pl.ds(mi * bm, bm), pl.ds((j - 1) * bn, bn)],
                    out_staging, wb_sem,
                )

            # ── EPILOGUE: drain AR for last tile ──
            last = accum_b if (n_tiles - 1) % 2 == 1 else accum_a
            if n_tiles == 1:
                last = accum_a

            _ar_ring_reduce(
                last, send_buf, recv_buf, ar_send_sem, ar_recv_sem,
                x_idx=x_idx, right_y=right_y, y_ring=y_ring,
            )
            _wb_to_hbm(
                last,
                out_ref.at[pl.ds(mi * bm, bm), pl.ds((n_tiles - 1) * bn, bn)],
                out_staging, wb_sem,
            )


def make_fused_ag_gemm_ar(
    x: jax.Array,
    w: jax.Array,
    *,
    bm: int = 512,
    bk: int = 512,
    bn: int = 512,
):
    m_local, k_local = x.shape
    _, n = w.shape

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=(1,),
        in_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # x
            pl.BlockSpec(memory_space=pl.ANY),  # w
        ],
        out_specs=[
            pl.BlockSpec(memory_space=pl.ANY),  # output
            pl.BlockSpec(memory_space=pl.ANY),  # recv weight workspace
            pl.BlockSpec(memory_space=pl.ANY),  # stashed weight (correct shard)
        ],
        scratch_shapes=(
            pltpu.SemaphoreType.DMA,             # ag_send_sem
            pltpu.SemaphoreType.DMA,             # ag_recv_sem
            pltpu.SemaphoreType.DMA,             # ar_send_sem
            pltpu.SemaphoreType.DMA,             # ar_recv_sem
            pltpu.SemaphoreType.DMA,             # wb_sem
            pltpu.VMEM((bm, bn), jnp.float32),  # send_buf
            pltpu.VMEM((bm, bn), jnp.float32),  # recv_buf
            pltpu.VMEM((bm, bn), jnp.bfloat16), # out_staging
        ),
    )

    out_shape = [
        jax.ShapeDtypeStruct((m_local, n), x.dtype),
        jax.ShapeDtypeStruct((k_local, n), w.dtype),  # recv workspace
        jax.ShapeDtypeStruct((k_local, n), w.dtype),  # stashed shard
    ]

    results = pl.pallas_call(
        functools.partial(
            _fused_ag_gemm_ar_kernel,
            bm=bm, bk=bk, bn=bn,
        ),
        grid_spec=grid_spec,
        out_shape=out_shape,
    )(x, w)

    return results[0]


def fused_ag_gemm_ar_kernel(inputs, weights):
    return make_fused_ag_gemm_ar(inputs, weights)


fused_ag_gemm_ar_fn = jax.jit(
    jax.shard_map(
        fused_ag_gemm_ar_kernel,
        mesh=mesh,
        in_specs=(P("x", "y"), P("x", None)),
        out_specs=P("x", None),
        check_vma=False,
    )
)

In [65]:
fused_ag_gemm_ar_fn(inputs, weights)

Array([[91.5, -233, -7.09375, ..., -17.625, 36.75, -173],
       [-75.5, 58.5, 85.5, ..., -80.5, 73, -66.5],
       [53, -19.5, 24.75, ..., -58.25, -175, 206],
       ...,
       [44.75, -112.5, 52.25, ..., -116.5, 150, 8.875],
       [-72.5, 284, -133, ..., -7.90625, -376, 124.5],
       [-396, -89.5, -3.25, ..., 92.5, -27.75, 11.1875]], dtype=bfloat16)

In [66]:
benchmark(fused_ag_gemm_ar_fn, inputs, weights)

BenchmarkResult (50 iters, 3 warmup)
  mean:       62.178 ms
  median:     62.177 ms
  stdev:       0.012 ms
  min:        62.155 ms
  max:        62.225 ms
  p95:        62.197 ms
  p99:        62.213 ms

In [67]:
ff = fused_ag_gemm_ar_fn(inputs, weights)
ff.block_until_ready()

with jax.profiler.trace('./traces/fully_fused'):
    ff = fused_ag_gemm_ar_fn(inputs, weights)
    ff.block_until_ready()

In [ ]:
"""
ppermute-x + local GEMM + psum-y (no full AG)
ppermute-x + local GEMM + fused y-reduction epilogue
fully fused stream: ICI->VMEM weight stream on x + tiled GEMM + incremental reduction on y
"""

In [ ]:
# NOTE: INCLUDE A VERSION WITH SOME XLA COMPILER FLAGS ENABLED TO TEST PERFORMANCE OR ASYNC ALL_GATHER, ETC.